In [2]:
from pathlib import Path
import pandas as pd
import json
import sys

root = Path(r"D:\Projects\speech-ai-learning")

if root not in sys.path:
    sys.path.insert(0, str(root))

from src.metrics import score_tokens
from src.evaluate import cer_tokens

#读取W7D6的预测结果
predictions_path = (
    root / "results" /"w07d6" /
    "whisper_small_baseline" / "predictions.csv"
)

predictions = pd.read_csv(predictions_path)

#找到目标样本
uid = "cv26_20558b6a5c8dbac9"

sample = predictions[
    predictions["uid"] == uid
]

assert len(sample) == 1

row = sample.iloc[0]

assert row["split"] == "dev"

#使用已经保存的规范化文本
reference_tokens = cer_tokens(
    row["score_reference"]
)

hypothesis_tokens = cer_tokens(
    row["score_hypothesis"]
)

#重新计算编辑操作
result = score_tokens(
    reference_tokens,
    hypothesis_tokens
)

# 创建分析记录
record = {
    "uid": uid,
    "split": "dev",
    "audio_path": row["audio_path"],
    "reference": row["reference"],
    "hypothesis": row["hypothesis"],
    "score_reference": row["score_reference"],
    "score_hypothesis": row["score_hypothesis"],
    "operations": json.dumps(
        result["operations"],
        ensure_ascii=False
    ),
    "S": result["S"],
    "D": result["D"],
    "I": result["I"],
    "C": result["C"],
    "N": result["N"],
    "distance": result["distance"],
    "CER": result["error_rate"],
    "error_type": "pending_verification",
    "verification_status": "pending_audio_review",
    "notes": "需要重新听取音频，确认Reference是否正确"
}

#保存
output_dir = root / "results" / "w07d7"
output_dir.mkdir(parents=True,exist_ok=True)

output_path = output_dir / "dev_error_analysis.csv"

#防止重复添加同一个UID
if output_path.exists():
    old = pd.read_csv(output_path)
    old = old[old["uid"] != uid]
    analysis = pd.concat(
        [old,pd.DataFrame([record])],
        ignore_index=True
    )
else:
    analysis = pd.DataFrame([record])

analysis.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Dev错误分析记录保存成功")
print(analysis[[
    "uid", "S", "D", "I", "C",
    "CER", "error_type", "verification_status"
]])

print("保存位置:", output_path)

Dev错误分析记录保存成功
                     uid  S  D  I   C   CER            error_type  \
0  cv26_20558b6a5c8dbac9  4  0  0  12  0.25  pending_verification   

    verification_status  
0  pending_audio_review  
保存位置: D:\Projects\speech-ai-learning\results\w07d7\dev_error_analysis.csv


In [3]:
analysis.loc[
    analysis["uid"] == uid,
    "error_type"
] = "model_error"

analysis.loc[
    analysis["uid"] == uid,
    "verification_status"
] = "verified"

analysis.loc[
    analysis["uid"] == uid,
    "notes"
] = "音频与Reference一致，模型存在4处字符替换"

analysis.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)